# compound_parser
Shared parser definition for all three simulations. Do **not** run simulations here: the simulation notebooks load this file with `%run`.

In [ ]:
!pip3 install -q pyactr

In [ ]:
"""
compound_parser.py — shared ACT-R (pyactr) DRS parser for noun-noun compound simulations.

"""
import re
import math
import numpy as np
import simpy
import pyactr as actr

## chunk types

In [ ]:
def define_chunktypes():
    actr.chunktype("parsing_goal",
                   "task stack1 stack2 stack3 stack4 arg_stack1 arg_stack2\
                    expected1 expected2 expected3 parsed_word\
                     dref_index th_index drs_index prev_drs_index")
    actr.chunktype("word","form cat pred")
    actr.chunktype("subdrs", "dref pred arg1 arg2 drs")
    actr.chunktype("link", "startnode endnode relation")

## model

In [ ]:
def build_parser(partial_matching=False, **overrides):
    """Create a fresh parser with chunk types, lexicon and all production rules.
    Link chunks are NOT added here: add them per simulation with add_link()
    and/or add_background_link()."""
    define_chunktypes()
    environment = actr.Environment(focus_position=(320, 180))
    params = dict(subsymbolic=True, retrieval_threshold=-4, latency_factor=0.15,
                  latency_exponent=0.05, emma_noise=False, rule_firing=0.011,
                  buffer_spreading_activation={"imaginal": 4}, strength_of_association=4,
                  association_only_from_chunks=False, spreading_activation_restricted=True,
                  activation_trace=True, partial_matching=partial_matching)
    params.update(overrides)
    parser = actr.ACTRModel(environment, **params)
    dm = parser.decmem
    parser.set_goal(name="imaginal", delay=0)
    parser.set_goal(name="discourse_context", delay=0)
    add_lexicon(dm)
    add_rules(parser)
    return parser, environment, dm

def add_lexicon(dm):
    theLexeme = actr.chunkstring(string="""
        isa     word
        form    the
        cat     Det
        """)
    dm.add(theLexeme, time=-10)

    aLexeme = actr.chunkstring(string="""
        isa     word
        form    a
        cat     Det
        """)
    dm.add(aLexeme, time=-10)

    stencilLexeme = actr.chunkstring(string="""
        isa     word
        form    stencil
        pred    STENCIL
        cat     N
        """)
    dm.add(stencilLexeme, time=-10)

    designLexeme = actr.chunkstring(string="""
        isa       word
        form      design
        pred      DESIGN
        cat       N
        """)
    dm.add(designLexeme, time=-10)

    wonLexeme = actr.chunkstring(string="""
        isa   word
        form  won
        pred  WON
        cat   V_intrans
        """)
    dm.add(wonLexeme, time=-10)

    crayonLexeme = actr.chunkstring(string="""
        isa       word
        form      crayon
        pred      CRAYON
        cat       N
        """)
    dm.add(crayonLexeme, time=-10)

    tunnelLexeme = actr.chunkstring(string="""
        isa       word
        form      tunnel
        pred      TUNNEL
        cat       N
        """)
    dm.add(tunnelLexeme, time=-10)

    roadLexeme = actr.chunkstring(string="""
        isa       word
        form      road
        pred      ROAD
        cat       N
        """)
    dm.add(roadLexeme, time=-10)

    doneLexeme = actr.chunkstring(string="""
        isa   word
        form  done
        pred  DONE
        cat   ADJ
        """)
    dm.add(doneLexeme, time=-10)

    lateLexeme = actr.chunkstring(string="""
        isa   word
        form  late
        pred  LATE
        cat   ADJ
        """)
    dm.add(lateLexeme, time=-10)

    isLexeme = actr.chunkstring(string="""
        isa   word
        form  is
        cat   Vcop
        """)
    dm.add(isLexeme, time=-10)

    butLexeme = actr.chunkstring(string="""
        isa   word
        form  but
        cat   CONJ
        """)
    dm.add(butLexeme, time=-10)


def add_link(dm, startnode, endnode, relation, every=120, since=3600):
    """Pre-load a link chunk acquired `since` seconds before the simulation and used every `every` seconds."""
    chunk = actr.chunkstring(string=f"""
        isa        link
        startnode  {startnode}
        endnode    {endnode}
        relation   {relation}
    """)
    dm.add(chunk, time=np.arange(-since, 0, every))
    return chunk

# ---------------------------------------------------------------------------
# Long-term (frequency-derived) base-level activation
# ---------------------------------------------------------------------------
# Optimized-learning approximation (Anderson & Lebiere 1998):
#     B ~= ln(n / (1 - d)) - d * ln(L)
# n = number of uses of the chunk, L = its lifespan in seconds (time since creation),
# d = decay. Intuition: when the exact times of past uses are unknown, treat them as
# spread evenly over the chunk's lifetime.
#
# NOTE: do NOT use pyactr's built-in `optimized_learning=True`. pyactr computes L as the
# time since the MOST RECENT presentation rather than since creation, and the switch is
# model-wide (it would change every chunk, including the explicitly timed ones).
# Instead, the value is computed here and attached with dm.add_activation(). pyactr
# combines it with any explicit timestamps inside the log:
#     B = ln( sum_k t_k^(-d)  +  exp(background) )

YEAR = 365 * 86400   # seconds

def background_activation(n, L=20 * YEAR, d=0.5):
    """Optimized-learning base level for n uses spread evenly over L seconds."""
    return math.log(n / (1 - d)) - d * math.log(L)

def add_background_link(dm, startnode, endnode, relation, p, N=1000, L=20 * YEAR, d=0.5):
    """Add a link chunk whose long-term history is a proportion `p` of `N` encounters
    with the compound over a lifespan of `L` seconds (same L for all links of a compound).
    Can be combined with add_link() on the same relation to add a recent discourse history."""
    chunk = actr.chunkstring(string=f"""
        isa        link
        startnode  {startnode}
        endnode    {endnode}
        relation   {relation}
    """)
    dm.add(chunk, time=np.array([]))        # in dm, no explicit timestamps
    dm.add_activation(chunk, background_activation(p * N, L, d))
    return chunk

def base_level(dm, chunk, now=0.0, d=0.5):
    """Base-level activation of `chunk` at time `now` (explicit timestamps + background)."""
    times = np.asarray(dm[chunk], dtype=float)
    times = times[times < now]
    total = np.sum((now - times) ** (-d)) if len(times) else 0.0
    bg = dm.activations.get(chunk)
    if bg is not None:
        total += math.exp(bg)
    return math.log(total)

## production rules

In [ ]:
def add_rules(parser):
    for i in range(1, 20):
        parser.productionstring(name="move dref index to x" + str(i+1), string="""
        =g>
        isa               parsing_goal
        task              move_dref_index
        dref_index        x"""+str(i)+"""
        ==>
        =g>
        isa               parsing_goal
        task              parsing
        dref_index        x"""+str(i+1))


    for i in range(1, 20):
        parser.productionstring(name="move drs index to u" + str(i+1), string="""
        =g>
        isa               parsing_goal
        task              move_drs_index
        drs_index         u"""+str(i)+"""
        drs_index         =drs_index
        ==>
        =g>
        isa               parsing_goal
        task              parsing
        prev_drs_index    =drs_index
        drs_index         u"""+str(i+1)+"""
    """)

    for i in range(1, 20):
        parser.productionstring(name="move th index to u" + str(i+1), string="""
        =g>
        isa               parsing_goal
        task              move_th_index
        th_index          k"""+str(i)+"""
        ==>
        =g>
        isa               parsing_goal
        task              parsing
        th_index          k"""+str(i+1)+"""
    """)

    parser.productionstring(name="move eyes", string="""
        =g>
        isa             parsing_goal
        task            ~reading_word
        task            ~finish_match_drs
        task            ~link_selection
        task            ~integrate_relation
        task            ~move_dref_index
        task            ~move_drs_index
        task            ~move_th_index
        task            ~parse_head
        task            ~parse_modifier
        ?retrieval>
        state           free
        =visual_location>
        isa             _visuallocation
        screen_x        =x
        screen_y        =ypos
        ==>
        =g>
        isa             parsing_goal
        task            attending_word
        ?visual_location>
        attended        False
        +visual_location>
        isa             _visuallocation
        screen_x        lowest
        screen_y        =ypos
    """, utility=-5)

    parser.productionstring(name="attend word", string="""
            =g>
            isa          parsing_goal
            task         attending_word
            =visual_location>
            isa         _visuallocation
            ?visual>
            state   free
            buffer  empty
            ?visual_location>
            state   free
            ==>
            =g>
            isa     parsing_goal
            task    reading_word
            +visual>
            isa     _visual
            cmd     move_attention
            screen_pos =visual_location
    """)

    parser.productionstring(name="encode word", string="""
        =g>
        isa             parsing_goal
        task            reading_word
        =visual>
        isa             _visual
        value           =w
        ==>
        =g>
        isa             parsing_goal
        task            encoding_word
        parsed_word     =w
        ~visual>
        ~retrieval>
    """)

    parser.productionstring(name="retrieve word", string="""
        =g>
        isa             parsing_goal
        task            encoding_word
        parsed_word     =w
        ==>
        +retrieval>
        isa             word
        form            =w
        =g>
        isa             parsing_goal
        task            retrieving_word
    """)

    parser.productionstring(name="shift and project word", string="""
        =g>
        isa             parsing_goal
        task            retrieving_word
        =retrieval>
        isa             word
        form            =w
        cat             =c
        ==>
        =g>
        isa             parsing_goal
        task            parsing
    """)

    parser.productionstring(name="finished: no visual input", string="""
        =g>
        isa             parsing_goal
        task            attending_word
        ?visual_location>
        state           error
        ==>
        ~g>
        ~discourse_context>
        ~imaginal>
    """)

    parser.productionstring(name="press key", string="""
        =g>
        isa             parsing_goal
        task            press_key
        =imaginal>
        isa             link
        startnode        None
        endnode          None
        relation         None

        ==>

        ~g>
        ~imaginal>
        +manual>
        isa             _manual
        cmd             'press_key'
        key             'F'
    """)

    parser.productionstring(name="compose: NP ==> Det N", string="""
        =g>
        isa             parsing_goal
        task            parsing
        stack1          S
        stack2          =s2
        dref_index      =dref_index
        drs_index       =drs_index
        =retrieval>
        isa             word
        cat             Det

        ==>

        =g>
        isa             parsing_goal
        task            move_dref_index
        stack1          N
        stack2          NP
        stack3          S
        stack4          =s2
        +discourse_context>
        isa             subdrs
        dref            =dref_index
        arg1            =dref_index
        drs             =drs_index
        ~retrieval>
    """)

    parser.productionstring(name="compose: N (empty imaginal)", string="""
        =g>
        isa             parsing_goal
        task            parsing
        stack1          N
        stack2          =s2
        stack3          =s3
        stack4          =s4
        =retrieval>
        isa              word
        cat              N
        pred             =p
        ?discourse_context>
        buffer           full
        ?imaginal>
        buffer           empty

        ==>

        =g>
        isa             parsing_goal
        task            parsing
        stack1          =s2
        stack2          =s3
        stack3          =s4
        =discourse_context>
        isa             subdrs
        pred            =p
        +imaginal>
        isa             link
        startnode       =p
    ~retrieval>
    """)

    parser.productionstring(name="compose: N (non-empty imaginal)", string="""
        =g>
        isa             parsing_goal
        task            parsing
        stack1          N
        stack2          =s2
        stack3          =s3
        stack4          =s4
        =retrieval>
        isa              word
        cat              N
        pred             =p
        ?discourse_context>
        buffer           full
        ?imaginal>
        buffer           full

        ==>

        =g>
        isa             parsing_goal
        task            parsing
        stack1          =s2
        stack2          =s3
        stack3          =s4
        =discourse_context>
        isa             subdrs
        pred            =p
        =imaginal>
        isa             link
        startnode       =p
    ~retrieval>
    """)

    parser.productionstring(name="compose: S ==> NP VP", string="""
        =g>
        isa             parsing_goal
        task            parsing
        stack1          NP
        stack2          S
        stack3          =s3
        stack4          =s4
        =discourse_context>
        isa             subdrs
        dref            =a1

        ==>

        =g>
        isa             parsing_goal
        task            parsing
        stack1          VP
        stack2          =s3
        stack3          =s4
        stack4          None
        arg_stack1      =a1
    """)

    parser.productionstring(name="compose: VP ==> Vcop ADJ", string="""
        =g>
        isa             parsing_goal
        task            parsing
        stack1          VP
        stack2          =s2
        stack3          =s3
        stack4          =s4
        drs_index       =drs_index
        arg_stack1      =a1
        =retrieval>
        isa             word
        cat             Vcop

        ==>

        =g>
        isa             parsing_goal
        task            parsing
        stack1          ADJ
        +discourse_context>
        isa             subdrs
        arg1            =a1
        drs             =drs_index
      ~retrieval>
      """)

    parser.productionstring(name="compose: ADJ", string="""
        =g>
        isa             parsing_goal
        task            parsing
        stack1          ADJ
        stack2          =s2
        stack3          =s3
        stack4          =s4
        drs_index       =drs_index
        arg_stack1      =a1
        =retrieval>
        isa             word
        cat             ADJ
        pred            =p

        ==>

        =g>
        isa             parsing_goal
        task            parsing
        stack1          =s2
        =discourse_context>
        isa             subdrs
        pred            =p
      ~retrieval>
      """)

    parser.productionstring(name="compose: VP ==> V_intrans", string="""
        =g>
        isa             parsing_goal
        task            parsing
        stack1          VP
        stack2          =s2
        stack3          =s3
        stack4          =s4
        drs_index       =drs_index
        arg_stack1      =a1
        =retrieval>
        isa             word
        cat             V_intrans
        pred            =p

        ==>

        =g>
        isa             parsing_goal
        task            parsing
        stack1          =s2
        stack2          =s3
        stack3          =s4
        stack4          None
        +discourse_context>
        isa             subdrs
        arg1            =a1
        pred            =p
        drs             =drs_index
      ~retrieval>
      """)

    parser.productionstring(name="compose: conjunction", string="""
        =g>
        isa             parsing_goal
        task            parsing
        stack1          S
        stack2          =s2
        stack3          =s3
        stack4          =s4
        drs_index       =drs_index
        arg_stack1      =a1
        =retrieval>
        isa             word
        cat             CONJ

        ==>

        =g>
        isa             parsing_goal
        task            move_drs_index
      ~retrieval>
      """)

    parser.productionstring(name="link selection", string="""
        =g>
        isa             parsing_goal
        task            link_selection
        =imaginal>
        isa             link
        startnode       =startnode
        endnode         =endnode

        ==>

        =g>
        isa             parsing_goal
        task            integrate_relation
        +retrieval>
        isa             link
        startnode       =startnode
        endnode         =endnode
      """)

    parser.productionstring(name="integrate_relation", string="""
        =g>
        isa             parsing_goal
        task            integrate_relation
        th_index        =th_index
        drs_index       =drs_index
        arg_stack1      =a1
        =retrieval>
        isa             link
        relation        =relation

        ==>

        =g>
        isa           parsing_goal
        task          move_th_index
        +discourse_context>
        isa            subdrs
        pred           =relation
        arg1           =th_index
        arg2           =a1
        drs            =drs_index
        =imaginal>
        isa            link
        relation       =relation
      ~retrieval>
      """)

    parser.productionstring(name="parse modifier", string="""
        =g>
        isa             parsing_goal
        task            parsing
        stack1          VP
        stack2          =s2
        stack3          =s3
        stack4          =s4
        th_index        =th_index
        drs_index       =drs_index
        =retrieval>
        isa             word
        cat             N
        ?discourse_context>
        buffer          full


        ==>

        =g>
        isa             parsing_goal
        task            parse_head
        =discourse_context>
        isa             subdrs
        arg1            =th_index
    """)

    parser.productionstring(name="parse head", string="""
        =g>
        isa             parsing_goal
        task            parse_head
        drs_index       =drs_index
        arg_stack1       =a1
        =retrieval>
        isa             word
        cat             N
        pred            =p
        =imaginal>
        isa             link
        startnode       =startnode
        endnode         =endnode

        ==>

        =g>
        isa             parsing_goal
        task            link_selection
        +discourse_context>
        isa             subdrs
        arg1            =a1
        pred            =p
        drs             =drs_index
        =imaginal>
        isa             link
        startnode       =p
        endnode         =startnode
    ~retrieval>
    """)

    parser.productionstring(name="compose: VP ==> V_trans", string="""
        =g>
        isa             parsing_goal
        task            parsing
        stack1          VP
        stack2          =s2
        stack3          =s3
        stack4          =s4
        drs_index       =drs_index
        dref_index      =dref_index
        arg_stack1      =a1
        =retrieval>
        isa             word
        cat             V_trans
        pred            =p

        ==>

        =g>
        isa             parsing_goal
        task            parsing
        stack1          S
        stack2          =s2
        stack3          =s3
        stack4          =s4
        +discourse_context>
        isa             subdrs
        arg1            =dref_index
        arg2            =a1
        pred            =p
        drs             =drs_index
      ~retrieval>
      """)

## run

In [ ]:
def run(parser, environment, sentence, finst=5):
    """Run the simulation on `sentence` (a string). Prints the full event trace."""
    parser.goals["g"].add(actr.chunkstring(string="""
        isa             parsing_goal
        task            reading_word
        stack1          S
        stack2          S
        dref_index      x1
        th_index        k1
        drs_index       u1
    """))
    words = sentence.split()
    parser.retrieval.finst = finst
    environment.current_focus = (160, 180)
    stimuli = [{x: {"text": w, "position": (160 + x * 40, 180)} for x, w in enumerate(words)}]
    parser.visualBuffer("visual", "visual_location", parser.decmem, finst=80)
    sim = parser.simulation(realtime=False, gui=False,
                            environment_process=environment.environment_process,
                            stimuli=stimuli, triggers="space")
    while True:
        try:
            sim.step()
        except simpy.core.EmptySchedule:
            break
    return sim

## similarities (simulation 3)

In [ ]:
NOUNS = ["stencil", "road", "design", "crayon", "tunnel"]

def set_glove_similarities(parser, nouns=NOUNS, scale=1.0, glove=None):
    """Match scores from GloVe (glove-wiki-gigaword-100): scale * (max(0, cos) - 1)."""
    if glove is None:
        import gensim.downloader as api
        glove = api.load("glove-wiki-gigaword-100")
    for i, a in enumerate(nouns):
        for b in nouns[i+1:]:
            cos = max(0.0, float(glove.similarity(a, b)))
            parser.set_similarities(a.upper(), b.upper(), scale * (cos - 1))
    return glove